---
title: "Car Insurance Claims"
author: "Andratx Bellmunt"
abstract: >
  Two different models -Poisson GLM and XGBoost- are used to predict the number of claims from car insurance data. Their performance is compared.
format:
  html:
    code-fold: false
    self-contained: true
    include-after-body: _tracker.html
jupyter: python3
number-sections: false
---

In [ ]:
from typing import List, Optional, Tuple

import pandas as pd
import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.io as pio

from sklearn.model_selection import train_test_split

In [ ]:
# Set renderer for plots
pio.renderers.default = "plotly_mimetype+notebook_connected"

# 1. Data Integrity & Preprocessing

After loading the raw data we split it right away into training, validation and test sets. All data preprocessing and exploratory data analysis (EDA) will use only the training set to avoid any leakage from the validation and test sets.

Since the data set contains car insurance claims it is expected that the target column `ClaimNb` has many zeroes and a long tail. To ensure the target mean frequency is consistent across the training, validation, and test partitions, we apply stratified sampling to the split. To that end we use a modified version of `ClaimNb` with a capping on the long tail. This ensures statistical balance while maintaining the integrity of rare high-count events in the target variable.

In [ ]:
# Load raw data (available at https://www.kaggle.com/datasets/floser/french-motor-claims-datasets-fremtpl2freq)
df_raw = pd.read_csv("../assets/freMTPL2freq.csv")

In [ ]:
def train_val_test_split(
    df: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # Define stratification column
    df["stratify_tmp"] = df["ClaimNb"].clip(upper=2)

    # Take 20% for testing
    df_train_val, df_test = train_test_split(
        df, test_size=0.2, random_state=13, stratify=df["stratify_tmp"]
    )

    # Split the remaining 80% into training and validation
    # (taking 25% of the 80% gives us a 60/20/20 overall split)
    df_train, df_val = train_test_split(
        df_train_val,
        test_size=0.25,
        random_state=13,
        stratify=df_train_val["stratify_tmp"]
    )

    # Drop temporary stratification column
    df_train.drop(columns="stratify_tmp", inplace=True)
    df_val.drop(columns="stratify_tmp", inplace=True)
    df_test.drop(columns="stratify_tmp", inplace=True)

    # Print number of rows of each chunk
    print(f"Training rows:   {len(df_train)}")
    print(f"Validation rows: {len(df_val)}")
    print(f"Test rows:       {len(df_test)}")

    return df_train, df_val, df_test

In [ ]:
df_raw_train, df_raw_val, df_raw_test = train_val_test_split(df_raw)

## Data cleaning

The following function applies all cleaning and preprocessing to the raw training data. The remaining of the section is devoted to explaining all the decisions we made in this regard.

In [ ]:
def clean_raw_data(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()

    # Set IDpol to integer type
    df["IDpol"] = df["IDpol"].astype(int)

    # Set VehAge to str type
    df["VehPower"] = df["VehPower"].astype(str)

    # Clip vehicle age
    df["VehAge"] = df["VehAge"].clip(upper=25)

    # Clip BonusMalus
    df["BonusMalus"] = df['BonusMalus'].clip(upper=150)

    # Clip number of claims
    df["ClaimNb"] = df["ClaimNb"].clip(upper=3)

    # Floor exposure (for offset and frequency calculations only)
    df["exposure_stable"] = df["Exposure"].clip(lower=1/12)

    return df


## Explaining cleaning choices

After initial exploration of the data we concluded that:

  - There are no null or unknown values that need treatment

  - `IDpol` can be safely converted to integer type

  - All categorical columns (`Area`, `VehBrand`, `VehGas`, `Region`) do not require any treatment since they have low cardinalities

  - `VehPower` transformed to string type so that models treat it as a category. Numerical order doesn't show a pattern with respect to claim frequency that justifies treating it differently.

  - `DrivAge` numerical column does not require treatment (all values are already between 18 and 100)

  - `VehAge` requires minimal treatment: we clip any value above 25 years old effectively creating a 25+ age (only 0.45% of the rows affected) that will account for all old cars.

  - `Density` numerical column does not need any treatment

  - The columns requiring more careful treatment are:

    - `ClaimNb`

    - `Exposure`

    - `BonusMalus` 

We shall include below the analysis and profiling that lead to the choices for these 3 columns (analysis for other columns is not included in this section but they are discussed in more depth during exploratory data analysis).

### ClaimNb

This is obviously a very relevant metric because it is the target value that our models will predict. Let us understand how it distributes:

In [ ]:
df_raw_train["ClaimNb"].value_counts().sort_index()

In [ ]:
print(f"Percentage of policies with no claims: {(len(df_raw_train[df_raw_train['ClaimNb']==0])/len(df_raw_train) * 100):.2f}%")

We have a zero inflated distribution where ~95% of the policies have no claims. The profile represents a classic car insurance distribution where a vast majority of the policies have no claims or a small number of them. That justifies considering Poisson like distributions to model them (this will be explored more extensively later on).

However, for a Poisson distribution with that many zeroes, drivers with many claims per year are a "statiscal black swan". We shall make our models focus on the underlying characteristics that lead to claims in general. In order to prevent them over-weighting features of these extreme outliers, we propose clipping the data to 3 or 4 claims maximum.

The expected effect is introducing a bit of bias to reduce variance drastically, improving the signal-to-noise ratio. A high number of claims can exert disproportionate influence on the loss function, potentially leading to over-fitting to volatility rather than learning the risk. By capping the target, we align the model's objective with the prediction of typical claim frequency, ensuring more stable and generalized results across the test set.

In order to make a final decision we shall measure the damage of clipping in terms of data loss:

In [ ]:
def compute_claims_clipping_damage(df: pd.DataFrame, clipval: int) -> None:
    lost_claims = df['ClaimNb'].sum() - df['ClaimNb'].clip(upper=clipval).sum()
    percent_lost = (lost_claims / df['ClaimNb'].sum()) * 100

    print(f"Claims clipping damage (clip={clipval}):")
    print(f"  Total claims removed by clipping: {lost_claims}")
    print(f"  Percentage of total claim volume lost: {percent_lost:.4f}%")
    print()

compute_claims_clipping_damage(df_raw_train, clipval=3)
compute_claims_clipping_damage(df_raw_train, clipval=4)

**Profile:** A "zero-inflated" distribution where ~95% of policies have no claims.

**Treatment:** Clip at 3

**Logic:** The jump from 3+ to 4+ claims represents less than 0.05% of the data so we choose the option that helps the most at reducing overdispersion. This makes the data more suitable for models based on Poisson distributions.

**Business interpretation:** This profile represents a classic car insurance distribution where a vast majority of the policies have no claims or a small number of them. That justifies considering Poisson like distributions.

### Exposure

`Exposure` is another very relevant column since together with `ClaimNb` it determines the claim frequency. Any Poisson based distribution model predicting number of claims shall use it as an offset.

The main risk comes from very short exposures that make frequency explode. Our treatment will aim at taming that effect.

In [ ]:
df_raw_train["Exposure"].describe()

In [ ]:
df_raw_train["Exposure"].hist(bins=24) # Each bin accounts for 1 month

In [ ]:
print(
    "Drivers with an exposure longer than a year (renewals): "
    f"{(len(df_raw_train[df_raw_train["Exposure"] > 1.0]) / len(df_raw_train) * 100):.2f}%"
)
print(
    "Drivers with an exposure shorter than a month (short term policies): "
    f"{(len(df_raw_train[df_raw_train["Exposure"] < 1 / 12]) / len(df_raw_train) * 100):.2f}%"
)

**Profile:** Significant bimodal distribution with peaks at <1 month and 12 months. ~17% of the data is short-term (<1 month). Only 0.19% of the data corresponds to drivers with exposures between 1 and 2 years.

**Treatment:** Floor at 1/12 (i.e. 1 month) for later frequency calculations; will be used as an offset/weighting for modeling.

**Logic:** Dropping 17% of the data would introduce survivorship bias. Flooring prevents "frequency explosions" (where 1 claim in 3 days appears as 100+ claims/year), allowing the models to learn from the entire population safely.

**Business interpretation:** Policies with less than 1 month exposure can be new business cancellations or cooling-off periods and hence have a different nature than policies with more exposure that represent an average customer. This will be addressed in more detail in the EDA.

### BonusMalus

Finally, `BonusMalus` is known to be a strong indicator for claim frequency in the car insurance industry. Since it is rated at a 50-350 scale but the break even (change from bonus to malus) is at 100 we shall treat it with care. Similarly to `ClaimNb`, we want to control for extreme outliers with very high malus that can add too much variance and deviate us from the main goal of modeling for most general type of claims.

In [ ]:
df_raw_train["BonusMalus"].describe()

In [ ]:
df_raw_train["BonusMalus"].hist(bins=50)

In [ ]:
print(
    "Drivers with perfect bonus (50): "
    f"{(len(df_raw_train[df_raw_train["BonusMalus"] == 50]) / len(df_raw_train) * 100):.2f}%"
)
print(
    "Drivers with malus (>100): "
    f"{(len(df_raw_train[df_raw_train["BonusMalus"] > 100]) / len(df_raw_train) * 100):.2f}%"
)
print(
    "Drivers with high malus (>150): "
    f"{(len(df_raw_train[df_raw_train["BonusMalus"] > 150]) / len(df_raw_train) * 100):.2f}%"
)

**Profile:** 56.73% of the data is pinned at the minimum score of 50. Only 1.14% of the data exists in the Malus zone (>100).

**Treatment:** Clip at 150

**Logic:** This preserves the high-risk signal of the malus drivers while preventing the 0.03% of extreme outliers from exerting undue leverage on the Poisson coefficients

**Business interpretation:** Most drivers have a perfect track record (50 bonus) and very few have malus. We shall see later that, as expected, this has a strong correlation with claim frequency

# 2. Exploratory Data Analysis

We shall perform the exploratory data analysis on the cleaned training set that we will use for modeling:

In [ ]:
df_train = clean_raw_data(df_raw_train)

We also define a series of functions that will help visualizing the data:

In [ ]:
def plot_actuarial_profile(
    dfa: pd.DataFrame, feature_type: str, feature: str, bins: Optional[List[int]]
) -> None:
    """Plot univariate distribution against frequency of claims"""
    assert feature_type in ["num", "cat"], "feature type must be 'num' or 'cat'"

    df = dfa.copy()

    # Create bins
    if feature_type == 'num':
        df['bin'] = pd.cut(df[feature], bins=bins)
    else:
        df['bin'] = df[feature]

    # Aggregate exposure and number of claims according to bins
    profile = df.groupby('bin', observed=True).agg(
        Exposure=('Exposure', 'sum'),
        Claims=('ClaimNb', 'sum')
    ).reset_index()

    # Sort
    profile = profile.sort_values('bin')

    # Now convert to string for the x-axis labels
    profile['bin_label'] = profile['bin'].astype(str)

    # Calculate Frequency
    profile['Frequency'] = profile['Claims'] / profile['Exposure']

    # Initialize figure
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Add Exposure as bars (volume)
    fig.add_trace(
        go.Bar(
            x=profile['bin_label'],
            y=profile['Exposure'],
            name="Total Exposure (Years)",
            marker_color='lightslategrey',
            opacity=0.6
        ),
        secondary_y=False,
    )

    # Add Frequency (risk)
    if feature_type == 'num':
        mode="lines+markers"
    else:
        mode="markers"
    fig.add_trace(
        go.Scatter(
            x=profile['bin_label'],
            y=profile['Frequency'],
            name="Claim Frequency",
            mode=mode,
            line=dict(color='firebrick', width=3)
        ),
        secondary_y=True,
    )

    # Styling
    fig.update_layout(
        title_text=f"<b>Actuarial Profile: {feature} vs Risk</b>",
        hovermode="x unified",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        yaxis2_range=[0, max(0.2, 1.1 * max(profile["Frequency"]))]
    )

    fig.update_xaxes(title_text=feature)
    fig.update_yaxes(title_text="Volume (Exposure years)", secondary_y=False)
    fig.update_yaxes(title_text="Annualized Frequency", secondary_y=True)

    fig.show()

In [ ]:
def plot_interaction_heatmap(
    dfa: pd.DataFrame,
    x_feature: str,
    x_bins: List[int],
    y_feature:str,
    y_bins: List[int]
) -> None:
    """Plot bivariate distributions against frequency of claims."""
    df = dfa.copy()

    # Create bins for both features
    df[f'{x_feature}_bin'] = pd.cut(df[x_feature], bins=x_bins)
    df[f'{y_feature}_bin'] = pd.cut(df[y_feature], bins=y_bins)

    # Aggregate Frequency
    interaction = (
        df
        .groupby([f'{x_feature}_bin', f'{y_feature}_bin'], observed=True)
        .agg(Freq=('ClaimNb', 'sum'), Exp=('Exposure', 'sum'))
        .reset_index()
    )

    # Convert to strings
    interaction[f'{x_feature}_bin'] = interaction[f'{x_feature}_bin'].astype(str)
    interaction[f'{y_feature}_bin'] = interaction[f'{y_feature}_bin'].astype(str)

    # Calculate Frequency
    interaction['Frequency'] = interaction['Freq'] / interaction['Exp']

    # Plot Heatmap
    fig = px.density_heatmap(
        interaction, 
        x=f'{x_feature}_bin', 
        y=f'{y_feature}_bin',
        z="Frequency",
        title=f"<b>Interaction: {x_feature} vs {y_feature} on Claim Frequency</b>",
        color_continuous_scale="Reds", 
        text_auto=".3f", # Shows 3 decimal places on the heatmap
        category_orders={
            x_feature: interaction[f'{x_feature}_bin'].unique().tolist(),
            y_feature: interaction[f'{y_feature}_bin'].unique().tolist()[::-1]
        },
        labels={f"{x_feature}_bin": x_feature, f"{y_feature}_bin": y_feature}
    )

    fig.show()

In [ ]:
def plot_exposure_by_claim_boxplot(df):
    "Boxplot to compare claims and exposure (together determine frequency of claims)."
    fig = px.box(
        df,
        x="ClaimNb",
        y="Exposure",
        title="<b>Exposure distribution by ClaimNb</b>",
        labels={"ClaimNb": "Number of Claims", "Exposure": "Policy Exposure (Years)"},
        points=False # Hides outliers to keep the chart clean and fast
    )

    fig.update_layout(
        showlegend=False,
        template="plotly_white",
        yaxis_range=[0, 2.1] # Focuses on our 0-2 year range
    )

    fig.show()

## Analysis

### BonusMalus

In [ ]:
plot_actuarial_profile(df_train, 'num', 'BonusMalus', bins=[49, 50, 75, 100, 125, 150])

The trend shows a sharp increase once we reach the malus zone. Creating an `is_malus` feature can help models discover this change.

### Driver Age

In [ ]:
plot_actuarial_profile(df_train, 'num', 'DrivAge', bins=[17, 25, 45, 65, 80, 100])

Again we see a hook-shaped trend. Here young drivers are detected to be significantly more prone to claims than more mature drivers. We can signal this fact to the models by creating an `is_young_driver` feature.

### Vehicle Age

In [ ]:
plot_actuarial_profile(df_train, 'num', 'VehAge', bins=[-1, 0, 5, 10, 15, 20, 25])

Similarly to young drivers, we can see how brand new cars (`VehAge`=0) get a much higher frequency of claims. After that, claims decline progressively as the vehicle gets older. Again, creating an `is_new_car` feature can be a useful signal for the models.

### Density

In [ ]:
plot_actuarial_profile(df_train, 'num', 'Density', bins=[0, 10, 100, 1000, 10000, 100000])

Note that we plotted `Density` at log scale and the frequency trend follows a fairly linear shape, especially for the most common densities (10 to 10000). In view of this fact we will create a new feature `log_density` to be used in our models.

### Vehicle Power / Gas / Brand

In [ ]:
for feature in ["Power", "Gas", "Brand"]:
    plot_actuarial_profile(df_train, "cat", f"Veh{feature}", None)

- For `VehPower` there are no major frequency differences among them. This is specially true for the most frequent ones (labels 4 to 7). It is not expected that this feature has much importance in the models unless there are hidden interactions with other features.

- `VehGas` has only two possible labels and both have similar levels of exposure and claim frequency. Similarly with what happens with power there is no direct evidence that this feature might be relevant at first glance.

- As for `VehBrand` there are no major differences but one of the most common brands, B12, is clearly the one with highest claim frequency. We shall see how much predictive power has that particular brand when modeling.

### Area / Region

In [ ]:
plot_actuarial_profile(df_train, 'cat', 'Area', None)
plot_actuarial_profile(df_train, 'cat', 'Region', None)

- `Area` seems to present a somewhat linear increase as we move from A to F. This may indicate some correlation with other metrics, possibly `log_density`. We shall explore it further below.

- `Region` does not present an obvious pattern or signal. There is no region with a lot of exposure that has a clearly higher or lower frequency of claims compared to other regions. We shall see how models handle region R94 since it has highest frequency but very little exposure.

In [ ]:
def area_logdensity_correlation(df:pd.DataFrame) -> float: 
    dict_aux = {"A": 1, "B": 2, "C": 3, "D":4, "E":5, "F": 6}
    ps_area = df["Area"].apply(lambda x: dict_aux[x])
    ps_logdensity =  np.log(df['Density'])
    correlation = np.corrcoef([ps_area, ps_logdensity])[0,1]

    return correlation

print(f"Correlation between Area and log_density: {area_logdensity_correlation(df_train):.2f}")

A correlation of 0.97 confirms that `Area` is a proxy of `log_density`, so we can safely drop it from the models since it will not add any extra relevant information. Of the two, it is better to keep `log_density` since it is continuous and more granular.

### DrivAge-BonusMalus interaction

In [ ]:
plot_interaction_heatmap(
    df_train, 
    "DrivAge", [17, 25, 45, 65, 80, 100], 
    "BonusMalus", [49, 50, 75, 100, 125, 150]
)

From univariate analysis we knew that young drivers and high malus have higher frequencies, so it was expected for the combination *young+malus* to have higher frequency. However, here we are seeing another kind of combination that is even riskier: *old+malus*. This interaction suggests that the low-risk benefit of an older driver is completely offset by the behavioral risk signaled by a high malus score. 

A model that is able to capture such kind of interactions, like gradient boosted decision trees, should perform well on the dataset.

### VehAge-BonusMalus interaction

In [ ]:
plot_interaction_heatmap(
    df_train, 
    "VehAge", [-1, 0, 5, 10, 15, 20, 25],
    "BonusMalus", [49, 50, 75, 100, 125, 150]
)

Here we see the exact same pattern that we saw for older driver age: older cars are not a risk on themselves, but they become a very high one one combined with malus.

### VehAge-DrivAge interaction

In [ ]:
plot_interaction_heatmap(
    df_train,
    "VehAge", [-1, 0, 5, 10, 15, 20, 25],
    "DrivAge", [17, 25, 45, 65, 80, 100]
)

After analyzing `BonusMalus` against `DrivAge` and `VehAge` and uncovering risk pockets that we wouldn't expect from univariate analysis alone, we see here that when we compare driver and vehicle age we do get what one would expect from the univariate analysis: young drivers in new cars are a higher risk combination.

Still, though, it is interesting to see how here the highest frequency is 0.385 when old people or old cars combined with high malus is in the order of 0.8 to 0.9, a much higher risk.

### Exposure by ClaimNb

In [ ]:
plot_exposure_by_claim_boxplot(df_train)

Demonstrating the relationship between `Exposure` and `ClaimNb`, justifying the use of the latter as an offset/weight:
  
  - Median shift: median exposure for 0 claims is much lower than when there are actual claims
  
  - Visual proof than the longer a policy is active (higher exposure) the more opportunity for a claim to occur
  
  - The first quartile of the 0 claims box highlights the large volume of short-term policies

## Feature engineering

In view of the EDA we add some binary helper columns that represent actuarial value:
  
  - `is_malus`: In many markets, being in the malus (penalty) zone triggers a different risk profile than just being a bad driver in the bonus zone.

  - `is_young_driver`: Drivers aged 18–25 represent a structural discontinuity in risk, often subject to different underwriting rules, second-driver exclusions, and higher behavioral volatility than the general population.

  - `is_new_car`: Brand-new cars often have a specific risk profile (higher frequency due to meticulous owners or lower due to better safety tech).

  - `is_short_term`: Very short policies are often associated with temporary or high-churn risks that act differently than standard annual policies.

We also create a new `log_density` feature, expected to have a linear relationship with the frequency of claims.

In [ ]:
def add_new_features(df: pd.DataFrame) -> pd.DataFrame:
    # Actuarial features
    df["is_malus"] = (df["BonusMalus"] > 100).astype(int)
    df["is_young_driver"] = (df["DrivAge"] <= 25).astype(int)
    df["is_new_car"] = (df["VehAge"] == 0).astype(int)
    df["is_short_term"] = (df["Exposure"] < 1 / 12).astype(int)

    # Log(Density)
    df["log_density"] = np.log(df["Density"])

    return df

# 3. Predictive Modeling

## Data preparation

In [ ]:
# Clean validation and test data sets
df_val = clean_raw_data(df_raw_val)
df_test = clean_raw_data(df_raw_test)

In [ ]:
# Add the new features
df_train = add_new_features(df_train)
df_val = add_new_features(df_val)
df_test = add_new_features(df_test)

In [ ]:
# Final list of features to be included
categorical_features = ["VehPower", "VehBrand", "VehGas", "Region"]      # Area is dropped as per the EDA
numerical_features = ["VehAge", "DrivAge", "BonusMalus", "log_density"]  # Density replaced by log_density as per the EDA
actuarial_features = ["is_malus", "is_young_driver", "is_new_car", "is_short_term"]
features = categorical_features + numerical_features + actuarial_features

## Model selection

In order to predict the number of claims (`ClaimNb`), we will test two distinct approaches:

1. **Poisson GLM** (with log-link function)
2. **XGBoost** (with Poisson objective)

While previously hinted at, **Poisson** models are the most suitable for this task: our goal is to count rare events, which is exactly what the Poisson distribution was designed for. As seen in the preprocessing phase, 95% of policies have zero claims, with counts declining rapidly; a distribution that fits a Poisson shape perfectly.

The EDA demonstrated that claim counts are tightly linked with `Exposure`, which defines the frequency of claims. This justifies using **exposure as an offset variable** in both models.

#### GLM

* **Justification:** Claim counts are discrete and sparse, making the Poisson distribution the natural statistical choice for frequency modeling.
* **Assumption:** The log-link function is used because insurance risk factors are typically multiplicative. The GLM allows us to quantify the exact "relativity" (e.g., a +15% risk increase) for each feature.
* **Strengths:** High transparency, regulatory acceptance, and mathematical stability.
* **Weaknesses:** It assumes features act independently and cannot detect complex interactions without manual intervention (e.g., the combined effect of high age and high malus).

#### XGBoost

* **Justification:** As a tree-based ensemble, XGBoost is "model-agnostic" regarding the data's shape and does not assume a linear relationship between log-frequency and features.
* **Capturing Interactions:** Its primary value lies in automatically discovering non-linear interactions and "local" patterns that a GLM would overlook.
* **Consistency:** By using the `count:poisson` objective and the same offset, we ensure a "fair fight" against the GLM, isolating the performance gain attributable to the algorithm's flexibility.

#### Why not Tweedie?

While Tweedie regression was considered, our target is the number of claims (frequency) rather than total cost (severity). Since counts are integers, the Poisson distribution (a special case of the Tweedie family where power p=1) is the most theoretically appropriate and parsimonious choice.

#### Benchmarking

It is good practice to use a simple model for benchmarking to quantify the value added by sophisticated algorithms. This also controls for preparation errors: if results are worse than a "dummy" model, a technical issue is likely. Our benchmark assigns the **average claim frequency** to every policy, essentially assuming "all drivers are equal."

#### Evaluation Metric

To evaluate performance, we must measure how much predictions deviate from actual values. We will use **Mean Poisson Deviance**, which is the standard loss metric for models fitting Poisson distributions.

## Benchmark model

In [ ]:
from sklearn.metrics import mean_poisson_deviance

In [ ]:
# Make predictions with benchmark model (assign average frequency to all drivers)
train_freq = df_train['ClaimNb'].sum() / df_train['exposure_stable'].sum()
df_train['bmk_pred'] = np.full(len(df_train), train_freq * df_train['exposure_stable'])
df_test['bmk_pred'] = np.full(len(df_test), train_freq * df_test['exposure_stable'])

# Benchmark reference
bmk_ref = mean_poisson_deviance(df_train["ClaimNb"], df_train['bmk_pred'])

# Benchmark performance
bmk_dev = mean_poisson_deviance(df_test['ClaimNb'], df_test['bmk_pred'])

print(f"Benchmark reference for training: {bmk_ref:.4f}")
print(f"Benchmark performance (test set): {bmk_dev:.4f}")

## GLM

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

### Training

In [ ]:
# Fit the Poisson GLM
glm_results = smf.glm(
    formula="ClaimNb ~ " + ' + '.join(features),
    data=df_train,
    family=sm.families.Poisson(),
    offset=np.log(df_train['exposure_stable']),
).fit(cov_type='HC3')

### Evaluation

In [ ]:
print(glm_results.summary())

In [ ]:
df_train['glm_pred'] = glm_results.predict(
    df_train, offset=np.log(df_train["exposure_stable"])
)

glm_ref = mean_poisson_deviance(df_train['ClaimNb'], df_train['glm_pred'])

print(f"Poisson deviance on training set: {glm_ref:.4f}")
print(f"Improvement over mean frequency: {(1-glm_ref/bmk_ref)*100:.2f}%")

**Model Fit**

The Poisson GLM reached convergence after 7 iterations. On the training set, the model achieved a mean Poisson deviance of 0.3071, capturing 5.55% of the deviance relative to the benchmark. The scale parameter was maintained at 1.0, ensuring the model adheres to the strict Poisson assumption required for a standard frequency rating structure.

**Statistical significance of engineered features**

- All engineered binary indicators —`is_malus`, `is_young_driver`, `is_new_car`, and `is_short_term`— yielded p-values of 0.000. This confirms that these features capture distinct risk discontinuities that a standard continuous model might smooth over.

**Main risk drivers**

Since the model uses a log-link, we can interpret the coefficients as multiplicative effects by exponentiating them:

   - `is_new_car` (*exp(0.9815)=2.6685*): Brand-new cars show a very high relative risk indicator at **+167%** compared to older cars. This suggests that new vehicles in this dataset are significantly more likely to report claims. This aligns with what we saw in the EDA.

   - `is_short_term` (*exp(0.8957)=2.4490*): Short-term policies are associated with a nearly **+145%** increase in frequency compared to standard annual policies, confirming they represent a much higher-churn, higher-risk segment.

   - `is_malus` (*exp(0.2171)=1.2425*): The split bonus/malus indicates that drivers with malus have a **+24%** claim frequency compared to the ones with bonus (note that each additional point in the `BonusMalus` score increases the expected claim frequency by approximately +2.1% as per its own coefficient)

   - `is_young_driver` (*exp(0.1215)=1.1292*): Even after controlling for the continuous `DrivAge`, being in the 18–25 bracket adds an additional **+13%** risk.

**Geographic & Technical Factors**

  - `log_density` (*exp(0.0403)=1.041*): As expected, higher population density correlates with higher frequency. The statistical relevance indicated by the p-value of 0.000 justifies our decision to keep this as a continuous predictor over the redundant `Area` categories.

  - `Region[T.R94]`(*exp(0.2093)=1.2328*): It is important to understand this example because region R94 exhibits one of the highest relativities in the model, with an estimated +23.3% increase in claim frequency relative to the baseline region. However, as noted in the EDA, R94 represents a small fraction of the total exposure. While the model captures this high observed frequency, the result should be interpreted with caution due to the limited sample size, which may lead to higher volatility in the estimate compared to high-exposure regions. Similarly, compared to other metrics that affect all drivers (e.g. everyone has a BonusMalus scoring but almost no one is in region R94), the importance of a single small region is limited.

  - `VehGas[T.Regular]` (*exp(0.0735)=1.0763*): Regular fuel types are statiscally significant (p-value=0.000) and show a **+7.6%** higher risk compared to the diesel. This an effect that was no apparent to us from the EDA alone. Qualitatively speaking, this figure may indicate that each type of car is generally utilised in different conditions or with different purposes, which may expose them differently to potential risks.

### Performance

In [ ]:
# Predict on test set
df_test["glm_pred"] = glm_results.predict(
    df_test, offset=np.log(df_test["exposure_stable"])
)

#### Calibration

In [ ]:
# Check there is no drift
print(f"Actual claims in test: {df_test['ClaimNb'].sum()}")
print(f"GLM predicted claims: {df_test['glm_pred'].sum():.2f}")

The fact that the total number of claims is properly matched shows that the model is mathematically sound and that using `Exposure` as an offset was the right move. Moreover the model is unbiased, meaning that if we were to use it for pricing, we would collect exactly enough premium to cover the predicted claims.

#### Deviance

In [ ]:
# Compute mean Poisson deviance
glm_dev = mean_poisson_deviance(df_test['ClaimNb'], df_test['glm_pred'])

print(f"Mean Poisson deviance: {glm_dev:.4f}")
print(f"Improvement over benchmark: {((1-glm_dev / bmk_dev) * 100):.2f}%")

We improved deviance by 5.48% with respect to the benchmark model. This confirms that the GLM model is strong and it means our features are capturing real, actionable risk signals.

## XGBoost

In [ ]:
import xgboost as xgb

### Data preparation

In [ ]:
# Convert to One-Hot Encoding
X_train = pd.get_dummies(df_train[features], drop_first=True)
X_val = pd.get_dummies(df_val[features], drop_first=True)
X_test = pd.get_dummies(df_test[features], drop_first=True)

# Align columns
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Create DMatrix structures for efficiency 
dtrain = xgb.DMatrix(X_train, label=df_train['ClaimNb'])
dval = xgb.DMatrix(X_val, label=df_val['ClaimNb'])
dtest = xgb.DMatrix(X_test, label=df_test['ClaimNb'])

# base_margin is the offset
dtrain.set_base_margin(np.log(df_train['exposure_stable']))
dval.set_base_margin(np.log(df_val['exposure_stable']))
dtest.set_base_margin(np.log(df_test['exposure_stable']))

### Hyperparameter tuning

Before running the model we perform hyperparameter selection by using randomized search with k-fold cross-validation. This is faster than grid search and simpler than Bayeasian search. It can also focus on more varied values of the most important parameters. The cross-validation ensures that the chosen parameters work consistenly across different subsets of the data.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, KFold

In [ ]:
# Define the model using the scikit-learn wrapper
# (necessary to use sklearn's SearchCV tools)
xgb_model = xgb.XGBRegressor(
    objective='count:poisson',
    tree_method='hist',
    n_estimators=150,  # A moderate number for the search phase
    n_jobs=-1
)

# Search space
param_grid = {
    'max_depth': [3, 4, 5, 6, 7],               # How complex are the interactions?
    'learning_rate': [0.01, 0.05, 0.1],         # How fast does it learn?
    'subsample': [0.7, 0.8, 0.9, 1.0],          # % of rows used per tree
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],   # % of features used per tree
    'reg_lambda': [1, 5, 10],                   # L2 regularization
    'reg_alpha': [0, 1, 5],                     # L1 regularization
    'gamma': [0, 0.5, 1, 2]                     # Prevents creating branches that are not informative
}

# Setup K-Fold and RandomizedSearch
kf = KFold(n_splits=5, shuffle=True, random_state=13)

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=15,                           # How many random combinations to try
    scoring='neg_mean_poisson_deviance', # We want to minimize Poisson deviance
    cv=kf,
    verbose=1,
    random_state=13,
    n_jobs=-1                            # Use all CPU cores
)

# Execute the search
random_search.fit(
    X_train, 
    df_train['ClaimNb'],
    base_margin=np.log(df_train['exposure_stable'])  # The sklearn wrapper of XGBoost requires passing the base_margin
)

print(f"Best Parameters: {random_search.best_params_}")

The combination of parameters that we got is safe and conservative set of parameters, which is what we need in insurance modeling to ensure the model generalizes well to new customers.

Let us briefly evaluate the results:

- `learning_rate: 0.01`, `max_depth: 6`: A very low learning rate combined with a moderate depth indicates a slow and steady learning process. This prevents the model from being overly influenced by individual unlucky observations, building a more robust consensus across trees.

- `reg_lambda: 5`, `reg_alpha: 5`, `gamma: 1`: These are relatively high values for L1 and L2 regularization. They penalize overly complex trees, forcing the model to only keep features that provide a significant, consistent boost to predictive power.

- `subsample: 0.7`: By training each tree on only 70% of the data, the model introduces "noise" that actually makes the final ensemble more resilient to outliers and sampling bias.

- `colsample_bytree: 1.0`: The model uses all available features for every tree. This suggests that in our specific dataset, the engineered features (like `is_young_driver` or `is_malus`) are distinct enough that the model doesn't need to hide them to find variety.

These results suggest that the XGBoost model is not just memorizing the training data. Because the learning rate is so low and the regularization is so high, the contest between XGBoost and GLM will be fair.


### Training

In [ ]:
params = {
    'objective': 'count:poisson',
    'tree_method': 'hist',             # Fast training for large data
    'eval_metric': 'poisson-nloglik',  # The Poisson equivalent of Log-Loss
    'seed': 13,                        # We want reproducible results
} | random_search.best_params_

In [ ]:
# Train with early stopping to prevent overfitting
xgb_results = xgb.train(
    params,
    dtrain,
    num_boost_round=5000,   # Large number or rounds because learning rate is small
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=250
)

In [ ]:
df_train['xgb_pred'] = xgb_results.predict(dtrain)

xgb_ref = mean_poisson_deviance(df_train['ClaimNb'], df_train['xgb_pred'])

print(f"Poisson deviance on training set: {xgb_ref:.4f}")
print(f"Improvement over mean frequency: {(1-xgb_ref/bmk_ref)*100:.2f}%")

The XGBoost model utilized a conservative learning rate (0.01) and early stopping with a 50-round patience window to ensure the ensemble prioritized generalizable patterns over training-set noise. It reached convergence after 4,276 iterations. The model achieved a Mean Poisson Deviance of 0.2860 on the training set, which captures 12.03% of variance with respect to the benchmark. The high number of boosting rounds required for convergence indicates that the model successfully navigated a complex loss surface to capture subtle, non-linear interactions across the feature set that are not accessible via standard additive modeling.

### Feature importance

In [ ]:
def create_feature_importance_df(xgb_results: xgb.core.Booster) -> pd.DataFrame:
    # Gain scores
    dict_scores = xgb_results.get_score(importance_type='gain')

    # Convert to DataFrame
    df_importance = pd.DataFrame(
        list(dict_scores.items()),
        columns=['feature', 'gain']
    ).sort_values(by='gain', ascending=False)

    # Calculate relative importance (%)
    total_gain = df_importance['gain'].sum()
    df_importance['importance_pct'] = (df_importance['gain'] / total_gain) * 100

    # Set feature as index for readability
    df_importance.set_index('feature', inplace=True)

    return df_importance

create_feature_importance_df(xgb_results)

The feature importances we are seeing for the XGBoost are very interesting, specially if we compare them to the ones we saw for GLM. Notice the following;

- `is_new_car`, `is_malus`, `is_young_driver` are not present altogether. How is this possible if, especially the first two, were the main features for the GLM model?
  
  - The explanation is that XGBoost is able to split the base features that we used to generate these derived features in a much more specific way.
  
  - Through `VehAge`, `BonusMalus` and `DrivAge` it already finds the binary actuarial values without us having to signal them. That's why it drops the derived features and and keeps only the base.
  
  - Note how both `VehAge` and `BonusMalus` are now high importance features, replicating what `is_new_car` and `is_malus` did for the GLM.

  - Note that this contrasts with the value `colsample_by_tree=1` that we got in parameter selection. However, we also got `alpha=5` for L1 regularization, which is prone to killing unneeded features.

- `is_short_term` is still a high importance feature. Why did it not drop it and use the base instead?
   
   - Because the base feature is `Exposure`, which is *not* a feature of our model, it is just the offset.
   
   - Using `Exposure` as offset has mathematical relevance, but the high importance of `is_short_term` shows that there is more to it: as observed in prior sections, there is an actuarial value that makes drivers with short exposures qualitatively different from other drivers.

- Interestingly `VehBrand_B12` now shows up as a very relevant player. In the EDA we already noticed the fact that it has the highest claim ratio among all brands and is among the most common brands. GLM apparently did not capture this importance too well because other brands had higher coefficients.

- As a final comment we see that `VehGas_Regular` also has high importance. In the GLM evaluation we discussed that it had higher risk than Diesel gas, but it was not placed that high in the feature importance list.

- It is also worth noticing that now `Region_R94` is placed much lower than it was for GLM. That would confirm the concerns we exposed in that section.

### Performance

In [ ]:
# Predict on test set
df_test['xgb_pred'] = xgb_results.predict(dtest)

#### Calibration

In [ ]:
# Check there is no drift
print(f"Actual claims in test set: {df_test['ClaimNb'].sum()}")
print(f"XGBoost predicted claims: {df_test['xgb_pred'].sum():.2f}")

Again the model has almost no drift with respect to the total number of claims, confirming that a Poisson objective with `Exposure` as the offset is indeed the right approach.

#### Deviance

In [ ]:
# Calculate Mean Poisson Deviance
xgb_dev = mean_poisson_deviance(df_test['ClaimNb'], df_test['xgb_pred'])

print(f"Mean Poisson deviance: {xgb_dev:.4f}")
print(f"Improvement over benchmark: {((1-xgb_dev / bmk_dev) * 100):.2f}%")

XGBoost improves the benchmark by a notable 9.08%. We shall compare it against the GLM in more detail later, but a jump from 5.48% to 9.08% improvement indicates that the model made a good job at capturing relevant non-linear interactions and combinations (e.g. the old_car+malus and old_driver+malus that we observed in the EDA).

However, while the improvement was 12.03% in the training set is now only 9.08% for the test set. For GLM the drop was only from 5.55% to 5.48%. This indicates that XGBoost may be overfitting a bit.

## Comparative evaluation of models

Let us make a reminder on how each of the models performed on the test set with respect to the benchmark model:

In [ ]:
print(f"Test set scores (mean Poisson deviance)")
print(40 * "=")
print(f"Benchmark   {bmk_dev:.4f}")
print(f"GLM         {glm_dev:.4f} (+{((1-glm_dev / bmk_dev) * 100):.2f}% improvement)")
print(f"XGBoost     {xgb_dev:.4f} (+{((1-xgb_dev / bmk_dev) * 100):.2f}% improvement)")

These results indicate that **XGBoost is a winner** at predicting claims for our car insurance claims.

To get a more exhaustive picture on how do the two models compare from an actuarial point of view, we shall use a couple of helpful visualizations.

In [ ]:
def get_lorenz_coords(actual: np.ndarray, predicted: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    # Sort actual values based on predictions (highest to lowest)
    order = np.argsort(predicted)[::-1]
    actual_sorted = actual.iloc[order].values

    # Cumulative sums
    cum_actual = np.cumsum(actual_sorted) / np.sum(actual_sorted)
    cum_pop = np.arange(1, len(actual) + 1) / len(actual)

    # Prepend (0,0)
    return np.insert(cum_pop, 0, 0), np.insert(cum_actual, 0, 0)

def plot_lorenz_curves(df: pd.DataFrame) -> None:
    # Calculate coordinates
    pop_glm, actual_glm = get_lorenz_coords(df['ClaimNb'], df['glm_pred'])
    pop_xgb, actual_xgb = get_lorenz_coords(df['ClaimNb'], df['xgb_pred'])
    pop_bmk, actual_bmk = get_lorenz_coords(df['ClaimNb'], df['bmk_pred'])

    # Plot
    fig = go.Figure()

    fig.add_trace(go.Scatter(x=pop_glm, y=actual_glm, name='GLM Lorenz Curve', line=dict(color='blue')))
    fig.add_trace(go.Scatter(x=pop_xgb, y=actual_xgb, name='XGBoost Lorenz Curve', line=dict(color='orange')))
    fig.add_trace(go.Scatter(x=pop_bmk, y=actual_bmk, name='Benchmark Lorenz Curve', line=dict(color='green')))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name='Random Selection (Null)', line=dict(color='black', dash='dash')))

    fig.update_layout(
        title="Lorenz Curve: Benchmark vs GLM vs XGBoost (Test Set)",
        xaxis_title="Cumulative Share of Population (Highest to Lowest Risk)",
        yaxis_title="Cumulative Share of Actual Claims Captured",
        template="plotly_white",
        height=600,
        width=800,
        xaxis_range=[0.0,1.0],
        yaxis_scaleanchor="x",
        yaxis_scaleratio=1
    )

    fig.show()

plot_lorenz_curves(df_test)

**Lorenz Curve** 

In our context the Lorenz curves aim at showing if a model is good at separating "safe" drivers from "dangerous" ones:

  - **The Diagonal Line:** Represents "random luck". If you pick 20% of drivers at random, you get 20% of the claims.

  - **The Model Curve:** If you rank drivers from highest to lowest risk, a good model will show that the top 20% of people (the ones the model flagged as high risk) actually account for, say, 50% of the claims.

  - **The Result:** As a consequence a curve that bows further away from the diagonal represents a better model.
  
Notice that our benchmark model is already better than a purely radom shuffle. That makes sense because the benchmark assumes "everyone is average", which is better than "anyone is anything". As observed by the scoring, GLM improves over the benchmark and XGBoost improves even more.

To put it in simple terms, the ranking criteria of each model is based on:

  - **Diagonal:** Random Shuffle

  - **Benchmark:** Ranked by Exposure

  - **GLM:** Ranked by Exposure + Linear Features

  - **XGBoost:** Ranked by Exposure + Complex Interactions

In [ ]:
def plot_double_lift_chart(df_test: pd.DataFrame) -> None:
    # Calculate ratio
    df_test['ratio'] = df_test['xgb_pred'] / df_test['glm_pred']

    # Sort into 10 buckets (deciles)
    df_test['decile'] = pd.qcut(df_test['ratio'], 10, labels=False)

    # Aggregate actual vs predicted frequency
    lift_data = df_test.groupby('decile').agg({
        'ClaimNb': 'sum',
        'exposure_stable': 'sum',
        'glm_pred': 'sum',
        'xgb_pred': 'sum'
    }).reset_index()

    lift_data['actual_freq'] = lift_data['ClaimNb'] / lift_data['exposure_stable']
    lift_data['glm_freq'] = lift_data['glm_pred'] / lift_data['exposure_stable']
    lift_data['xgb_freq'] = lift_data['xgb_pred'] / lift_data['exposure_stable']

    # Plot
    fig = go.Figure()

    fig.add_trace(go.Bar(x=lift_data['decile'], y=lift_data['actual_freq'], name='Actual Frequency', marker_color='lightgrey'))
    fig.add_trace(go.Scatter(x=lift_data['decile'], y=lift_data['glm_freq'], name='GLM Pred Frequency', line=dict(color='blue', width=3)))
    fig.add_trace(go.Scatter(x=lift_data['decile'], y=lift_data['xgb_freq'], name='XGBoost Pred Frequency', line=dict(color='orange', width=3)))

    fig.update_layout(
        title="Double Lift Chart: Deciles of (XGB / GLM) Ratio",
        xaxis_title="Decile (1=GLM overestimates relative to XGB, 10=XGB overestimates relative to GLM)",
        yaxis_title="Claim Frequency (actual)",
        template="plotly_white",
        height=600,
        barmode='group'
    )

    fig.show()

plot_double_lift_chart(df_test)

**Double Lift Chart**

This chart aims at answering a very simple actuarial question: "If my competitor uses a GLM and I use XGBoost, who wins?"

How to read it:

- Going from left to right we move progressively from Bucket 0-where GLM thinks the risk is much higer than XGBoost-to Bucket 9-where XGBoost thinks the risk is much higer than GLM.

- The height of each bar is the actual claim frequency on each bucket.

- The blue and yellow lines are the predicted frequencies for each of the models. A marker above the bar is an overstimate. A marker below the top of the bar is an understimate.

- Therefore, a model performs better if it "hugs" the gray bars (the markers are at a similar height as the top of the bars).

In this regard, XGBoost clearly made a much better job than GLM at capturing the actual frequencies.

## Residual Analysis and Irreducible Risk

Before wrapping up our work we will take a look at residual analysis by inspecting the top 25 errors of the three models (we include the benchmark because it provides useful insights). 

The number 25 is not chosen lightly: this is the exact number of rows in our test set that have 3 claims, corresponding to the capping we put on `ClaimNb`.

In [ ]:
def get_post_mortem_df(
    df: pd.DataFrame, model_name: str, relevant_features: List[str]
) -> pd.DataFrame:
    # Calculate absolute error on the Test Set
    df['error'] = np.abs(df['ClaimNb'] - df[f'{model_name}_pred'])

    # Get the top 10 most "surprising" rows
    top_errors = df.sort_values(by='error', ascending=False).head(25)

    # Set IDpol as index
    top_errors.set_index('IDpol', inplace=True)

    # Display relevant columns to look for patterns
    cols_to_check = ['ClaimNb', f'{model_name}_pred', 'Exposure'] + relevant_features

    return top_errors[cols_to_check]

In [ ]:
bmk_pm = get_post_mortem_df(df_test, "bmk", features)
glm_pm = get_post_mortem_df(df_test, "glm", features)
xgb_pm = get_post_mortem_df(df_test, "xgb", features)

In [ ]:
# Count how many errors are shared for each pair of models
print(f"Shared errors in top 25 (Benchmark vs GLM):     {len(set(bmk_pm.index).intersection(set(glm_pm.index)))}")
print(f"Shared errors in top 25 (Benchmark vs XGBoost): {len(set(bmk_pm.index).intersection(set(xgb_pm.index)))}")
print(f"Shared errors in top 25 (GLM vs XGBoost):       {len(set(glm_pm.index).intersection(set(xgb_pm.index)))}")

In [ ]:
# Count how many claims do the top 25 errors have for each model
display(bmk_pm['ClaimNb'].value_counts().to_frame())
display(glm_pm['ClaimNb'].value_counts().to_frame())
display(xgb_pm['ClaimNb'].value_counts().to_frame())

Note that all models report the same top 25 errors and that they all struggle at capturing rows with a top number of claims. This fact should be read from two different angles:

**Model limitations**

A Poisson model (including GLM and XGBoost) predicts the expected value. Even for the riskiest driver in our dataset, the model might only predict relatively low frequency (e.g. the Benchmark predicts a frequency of ~0.1 for all drivers):

  - When that driver has 0 claims in a year, the error is small (0.1).

  - When that driver has 3 claims in a year, the gap is massive (2.9) and even more if exposure was shorter.

Since the models are grounded in reality, they refuse to predict a "3" for anyone, because no one has a *300% probability* of a claim. Therefore, the rows with 3 claims are always the top errors by definition.

**Stochastic noise in the data**

The fact that these are the top 25 errors for *all three models* proves that these 3-claim events are stochastic rather than systematic.

  - If the XGBoost had "fixed" these errors while the GLM didn't, it would mean there was a hidden pattern (e.g., "Drivers from Region X always have 3 claims")
  
  - Since the XGBoost also failed to predict them, it confirms there is no feature in our data (Age, Brand, Gas, etc.) that reliably points to a triple-claim outcome. Moreover, it proves that XGBoost hasn't hallucinated a patter where non exists.

On an extra note, inspecting the errors in more detail reveals that the policies with IDs *2247986*, *2248174* and *2277762* are perfect duplicates (except for their IDs) which might be due to some accidental data entry. In a real world scenario it would be worth double checking extra information on specific policies -if available- to confirm what is the exact situation.

# 4. Conclusions & Recommendations

### Key Findings

* **Performance:** The **XGBoost model** is the clear winner, explaining **9.08%** of the deviance on unseen data, clearly improving the predictive power of the GLM (**5.48%**).

* **Segmentation:** The Lorenz Curve confirms that XGBoost is significantly more capable of isolating high-risk segments, which allows for more competitive pricing and better risk selection.

* **Model Reliability:** While slight overfitting was observed in the XGBoost model, it remains robust. The "top errors" analysis reveals that remaining inaccuracies are due to random claim volatility (3-claim outliers) rather than model failure, affecting all models equally.

### Recommendations

* **Adopt XGBoost for Pricing:** Transition from the GLM to the XGBoost framework to capture non-linear risks that traditional models miss.

* **Monitor "Black Swan" Outliers:** Since 3-claim events are currently unpredictable, consider a separate "Large Loss" loading or reinsurance strategy, as these events are stochastic and not driven by the current feature set.

* **Data Enrichment:** To further close the gap on the 9% test deviance, investigate new features (e.g., annual mileage or telematics) that might explain the high-frequency outliers that the current model classifies as "safe".